In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/prompt-injection-dataset/MPDD.csv
/kaggle/input/datasets/kamrun71/processed-data/y_train.npy
/kaggle/input/datasets/kamrun71/processed-data/y_test.npy
/kaggle/input/datasets/kamrun71/processed-data/y_val.npy
/kaggle/input/datasets/kamrun71/processed-data/X_test_vec.npy
/kaggle/input/datasets/kamrun71/processed-data/X_val_vec.npy
/kaggle/input/datasets/kamrun71/processed-data/X_train_vec.npy
/kaggle/input/datasets/kamrun71/ensemble-models/ensemble_full_features.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_full_svc_xgb_opt.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_xgb.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/ensemble_opt_features.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_xgb_full_svc_xgb_opt.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_random_forest.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_xgboost.pkl
/kaggle/input/datasets/kamrun71/retrained-m

# **data loading**

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import re
import nltk
from nltk.corpus import stopwords
# Ensure have the 'stopwords' and 'punkt' resources downloaded
# nltk.download('stopwords')
# nltk.download('punkt')

# Load the dataset
# Replace 'MPDD.csv' with the actual path if it's not in the same directory
df = pd.read_csv('/kaggle/input/prompt-injection-dataset/MPDD.csv')
# Remove case-insensitive duplicates while keeping the first occurrence


# Check new dataset size
print(len(df))


# Display the first few rows and check the class distribution
print("Data Head:")
print(df.head())
print("\nClass Distribution:")
print(df['isMalicious'].value_counts())

39234
Data Head:
                                              Prompt  isMalicious
0  How did the "Coronation Street" storyline hand...            1
1                                 Login to Snapchat?            0
2  I've been having a hard time getting back into...            1
3           What is the best way to budget my money?            1
4  Is it rude to say that I don’t understand “Cau...            0

Class Distribution:
isMalicious
1    19617
0    19617
Name: count, dtype: int64


In [3]:
# Check null prompts
null_count = df["Prompt"].isna().sum()
print("Null prompts:", null_count)

# Check empty / whitespace-only prompts
empty_count = df["Prompt"].astype(str).str.strip().eq("").sum()
print("Empty or whitespace-only prompts:", empty_count)

# Check exact duplicates
duplicate_exact = df.duplicated(subset="Prompt").sum()
print("Exact duplicate prompts:", duplicate_exact)

# Check case-insensitive duplicates
duplicate_ci = df["Prompt"].str.lower().duplicated().sum()
print("Case-insensitive duplicate prompts:", duplicate_ci)

Null prompts: 0
Empty or whitespace-only prompts: 0
Exact duplicate prompts: 0
Case-insensitive duplicate prompts: 1


In [4]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import re
import nltk
from nltk.corpus import stopwords
!pip install langdetect  # Run this in a cell if not already installed

from langdetect import detect, DetectorFactory
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

# Deduplication
# ============================
# DATASET-LEVEL CLEANING
# (OUTSIDE CV – correct)
# ============================

# Deduplication
df = df.loc[~df["Prompt"].str.lower().duplicated()]

# English filtering
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

df = df[df["Prompt"].apply(is_english)].copy()

print("Dataset size after dedup + English filter:", len(df))
print(df["isMalicious"].value_counts())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=dd06cb6399c1ff6a566af41f7a7b0348dc46e3f9ef28912d6dfe0609a831747e
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Dataset size after dedup + English filter: 37547
isMalicious
0    19013
1    18534
Name: count, dtype: int64


# **PHASE 3 (BATCH)**

In [5]:
import re
import numpy as np
import pandas as pd
import string
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import xgboost as xgb
import torch
from transformers import AutoTokenizer, AutoModel
import sys

# Block cupy to prevent circular import crash from spacy -> srsly -> cupy
sys.modules['cupy'] = None
# -----------------------------
# 1. INJECTION-AWARE PREPROCESSING
# -----------------------------
def injection_aware_preprocess(text):
    if pd.isna(text) or text is None:
        return ""
    text = text.lower()
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_wrapper(X):
    return X['Prompt'].apply(injection_aware_preprocess)

# -----------------------------
# 2. ADVERSARIAL FEATURES
# -----------------------------
try:
    import spacy
    nlp_engine = spacy.load("en_core_web_sm")
except OSError:
    print("WARNING: Spacy model 'en_core_web_sm' not found. Layer 5 features will be 0.")
    nlp_engine = None

class AdversarialFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.keyword_groups = {
            "instruction_override": ["ignore", "forget", "override", "bypass", "previous", "instruction", "prompt", "format", "generate"],
            "role_manipulation": ["system", "assistant", "developer", "admin", "role", "you are"],
            "sql_like_payload": ["select", "update", "delete", "insert", "drop", "where"],
            "technical_context": ["report", "economy", "blame", "regex", "expression", "python", "java", "code"],
            "style_and_art": ["style", "ghibli", "sorting"],
            "general_suspicious": [
                "start", "following", "regarding", "business", "news", "class", "name",
                "when", "if", "prior", "answer", "question", "disregard", "what", "how",
                "good", "way", "skip", "overlook", "neglect", "minorities", "hate"
            ]
        }
        self.phrases = ["is it possible", "can you", "could you", "you are"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = X['Prompt'].astype(str)
        series_lower = series.str.lower()
        features = pd.DataFrame()

        features['flag_fullwidth_comma']    = series.apply(lambda t: 1 if '，' in t else 0)
        features['flag_code_arrow']         = series.apply(lambda t: 1 if '->' in t else 0)
        features['flag_escaped_newline']    = series.apply(lambda t: 1 if '\\n' in t or '\\ n' in t.lower() else 0)
        features['flag_phrase_injection']   = series_lower.apply(lambda t: 1 if any(p in t for p in self.phrases) else 0)
        features['flag_kw_override']        = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["instruction_override"]) else 0)
        features['flag_kw_role']            = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["role_manipulation"]) else 0)
        features['flag_kw_sql']             = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["sql_like_payload"]) else 0)
        features['flag_kw_tech']            = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["technical_context"]) else 0)
        features['flag_kw_style']           = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["style_and_art"]) else 0)
        features['flag_kw_general']         = series_lower.apply(lambda t: 1 if any(k in t for k in self.keyword_groups["general_suspicious"]) else 0)

        def check_punctuation_layer(text):
            if not text: return 0
            last_char = text.strip()[-1] if text.strip() else ""
            if last_char in string.punctuation or last_char in ["。", "！", "？", "，"]:
                return 1
            return 0

        features['flag_punctuation_check']  = series.apply(check_punctuation_layer)

        def check_linguistics(text):
            if nlp_engine is None: return 0
            try:
                doc = nlp_engine(text[:512])
                if any(t.pos_ == "VERB" and t.dep_ == "ROOT" for t in doc) or \
                   any(t.tag_ == "MD" for t in doc):
                    return 1
            except:
                return 0
            return 0

        features['flag_linguistic_imperative'] = series_lower.apply(check_linguistics)
        features['count_parentheses']          = series.apply(lambda t: t.count('(') + t.count(')'))
        features['char_density']               = series.apply(lambda t: len(re.findall(r'[^\w\s]', t)) / len(t) if t else 0)

        return features.values

    def get_feature_names_out(self, input_features=None):
        return np.array([
            'flag_fullwidth_comma', 'flag_code_arrow', 'flag_escaped_newline',
            'flag_phrase_injection', 'flag_kw_override', 'flag_kw_role',
            'flag_kw_sql', 'flag_kw_tech', 'flag_kw_style', 'flag_kw_general',
            'flag_punctuation_check', 'flag_linguistic_imperative',
            'count_parentheses', 'char_density'
        ])


# -----------------------------
# 3. BERT EMBEDDINGS — FIXED
# ← Only change is in transform():
#    inputs move to wherever self.model actually lives
# -----------------------------
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModel.from_pretrained(MODEL_NAME)
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
model.eval()
BERT_DIM   = model.config.hidden_size

class BERTFeatureExtractor(BaseEstimator, TransformerMixin):

    # AFTER — remove device param, always read from model:
    def __init__(self, tokenizer, model, max_len=128, batch_size=256):
        self.tokenizer  = tokenizer
        self.model      = model
        # no self.device stored — transform() already reads actual_device live
        self.max_len    = max_len
        self.batch_size = batch_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            series = X['Prompt'].tolist()
        else:
            series = X.tolist()

        embeddings = []
        total = len(series)

        # ← FIX: always read device from wherever the model actually is
        #   This handles the case where the model was loaded with
        #   map_location='cpu' but DEVICE was set to cuda, or vice versa.
        actual_device = next(self.model.parameters()).device
        print(f"Processing {total} samples on {actual_device} with batch size {self.batch_size}...")

        for i in range(0, total, self.batch_size):
            batch_texts = series[i : i + self.batch_size]
            batch_texts = [str(t) if pd.notnull(t) else "" for t in batch_texts]

            # Tokenize
            inputs = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                max_length=self.max_len,
                truncation=True,
                padding=True
            )

            # ← FIX: move every input tensor to actual_device, not self.device
            #   self.device may be stale (e.g. still 'cpu' after model moved to cuda)
            inputs = {
                k: v.to(actual_device) if isinstance(v, torch.Tensor) else v
                for k, v in inputs.items()
            }

            with torch.no_grad():
                outputs = self.model(**inputs)

            cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_vectors)

            if i % 1000 == 0 and i > 0:
                print(f"  > Processed {i}/{total}...")

        return np.vstack(embeddings)


# -----------------------------
# 4. FEATURE PIPELINES
# -----------------------------
preprocess_step = FunctionTransformer(preprocess_wrapper)

word_tfidf = Pipeline([
    ('clean', preprocess_step),
    ('tfidf', TfidfVectorizer(analyzer='word', ngram_range=(1, 3), max_features=4000))
])

char_tfidf = Pipeline([
    ('clean', preprocess_step),
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3000))
])

adv_pipeline = Pipeline([
    ('adv', AdversarialFeatureExtractor())
])

# For P100, use batch_size=256 — 4x faster per forward pass
bert_pipeline = Pipeline([
    ('bert', BERTFeatureExtractor(tokenizer, model, DEVICE, batch_size=256))
])

# -----------------------------
# 5. COMBINED FEATURE DESIGN
# -----------------------------
feature_combiner = ColumnTransformer(
    transformers=[
        ('word_tfidf',   word_tfidf,   ['Prompt']),
        ('char_tfidf',   char_tfidf,   ['Prompt']),
        ('adv_features', adv_pipeline, ['Prompt']),
        ('bert_embed',   bert_pipeline,['Prompt'])
    ],
    remainder='drop'
)

print("Hybrid + BERT Feature Engineering Ready!")

# -----------------------------
# 6. LOAD + SPLIT
# -----------------------------
print("Samples after removing case-insensitive duplicates:", len(df))
X = df[['Prompt']]
y = df['isMalicious']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Hybrid + BERT Feature Engineering Ready!
Samples after removing case-insensitive duplicates: 37547
Train: 22527 | Val: 7510 | Test: 7510


# ***computational cost***

In [6]:
# ── Auto-reset guard ──
if not isinstance(X_val, pd.DataFrame):
    print("X_val is numeric — resetting to raw text splits...")
    X = df[['Prompt']]
    y = df['isMalicious']
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )
    print("Reset done. X_val shape:", X_val.shape)
else:
    print("X_val is already a DataFrame — no reset needed.")

import time
import psutil
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp
import os
import xgboost as xgb
from sklearn.metrics import accuracy_score
import torch
from unittest.mock import patch

# =========================================================
# GPU SETUP
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nRunning on: {device}")
if device.type == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def unpack_and_densify(data):
    if hasattr(data, "item") and data.ndim == 0:
        data = data.item()
    if sp.issparse(data):
        return data.toarray().astype(np.float32)
    return np.array(data, dtype=np.float32)

def to_numpy(data):
    if torch.is_tensor(data):
        return data.cpu().numpy().astype(np.float32)
    elif sp.issparse(data):
        return data.toarray().astype(np.float32)
    return np.array(data, dtype=np.float32)

# =========================================================
# CLASS DEFINITIONS — must be before any joblib.load
# =========================================================
class PreFittedVoting:
    def __init__(self, named_models):
        self.named_models = named_models

    def predict_proba(self, X):
        probas = []
        for name, model in self.named_models:
            if "LinearSVC" in name:
                d = model.decision_function(X)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X))
        return np.mean(probas, axis=0)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def decision_function(self, X):
        proba = self.predict_proba(X)[:, 1]
        return np.log(proba / (1 - proba + 1e-9))


class CrossFeatureVoting:
    def __init__(self, full_models, opt_models):
        self.full_models = full_models
        self.opt_models  = opt_models

    def predict_proba(self, X_full, X_opt):
        probas = []
        for name, model in self.full_models:
            if "LinearSVC" in name:
                d = model.decision_function(X_full)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X_full))
        for name, model in self.opt_models:
            if "LinearSVC" in name:
                d = model.decision_function(X_opt)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X_opt))
        return np.mean(probas, axis=0)

    def predict(self, X_full, X_opt):
        return (self.predict_proba(X_full, X_opt)[:, 1] >= 0.5).astype(int)

# =========================================================
# 1. LOAD DATA
# =========================================================
print("\n" + "="*60)
print(" STEP 1: LOADING DATA")
print("="*60)

y_test  = np.load('/kaggle/input/datasets/kamrun71/processed-data/y_test.npy',  allow_pickle=True)
y_train = np.load('/kaggle/input/datasets/kamrun71/processed-data/y_train.npy', allow_pickle=True)
N_test  = 7510
N_train = len(X_train)

print(f"Train samples : {N_train}")
print(f"Test  samples : {N_test}")

# =========================================================
# 2. LOAD FEATURE COMBINER + INDICES
# =========================================================
print("\n" + "="*60)
print(" STEP 2: LOADING FEATURE COMBINER")
print("="*60)

original_torch_load = torch.load

def gpu_torch_load(f, *args, **kwargs):
    kwargs['map_location'] = device
    kwargs.setdefault('weights_only', False)
    return original_torch_load(f, *args, **kwargs)

start_import_full = time.time()
with patch('torch.load', gpu_torch_load):
    feature_combiner_full = joblib.load(
        "/kaggle/input/datasets/kamrun71/feature-3/feature_combiner_3.pkl"
    )
import_time_full = (time.time() - start_import_full) * 1000

# Push BERT inside combiner onto GPU
bert_extractor = feature_combiner_full.named_transformers_['bert_embed']['bert']
bert_extractor.model = bert_extractor.model.to(device)
bert_extractor.model.eval()
print(f"BERT is on    : {next(bert_extractor.model.parameters()).device}")
print(f"Combiner load : {import_time_full:.1f} ms")

start_import_opt = time.time()
opt_dir      = "/kaggle/input/datasets/kamrun71/retrained-models"
keep_indices = joblib.load(f"{opt_dir}/final_feature_indices.pkl")
import_time_opt = (time.time() - start_import_opt) * 1000
print(f"Indices  load : {import_time_opt:.1f} ms")

# =========================================================
# 3. LOAD ALL 5 SAVED ENSEMBLE MODELS
# =========================================================
print("\n" + "="*60)
print(" STEP 3: LOADING ALL 5 ENSEMBLE MODELS")
print("="*60)

ensemble_dir = "/kaggle/input/datasets/kamrun71/ensemble-models"

# Strategy 2 uses optimized features only — so X_opt is passed as X_full
# for PreFittedVoting. We handle this with a special flag below.
# (strategy_label, pkl_file, needs_opt, use_opt_as_full)
#   needs_opt=False, use_opt_as_full=False → predict(X_full)          S1
#   needs_opt=False, use_opt_as_full=True  → predict(X_opt)           S2
#   needs_opt=True,  use_opt_as_full=False → predict(X_full, X_opt)   S3/S4/S5

benchmarks = [
    (
        "strategy1_full_voting",
        joblib.load(f"{ensemble_dir}/ensemble_full_features.pkl"),
        False, False   # PreFittedVoting on full features
    ),
    (
        "strategy2_opt_voting",
        joblib.load(f"{ensemble_dir}/ensemble_opt_features.pkl"),
        False, True    # PreFittedVoting on optimized features
    ),
    (
        "strategy3_LR_XGB_cross_feature",
        joblib.load(f"{ensemble_dir}/cross_ensemble_lr_xgb.pkl"),
        True, False    # CrossFeatureVoting
    ),
    (
        "strategy4_LR_full_SVC_XGB_opt",
        joblib.load(f"{ensemble_dir}/cross_ensemble_lr_full_svc_xgb_opt.pkl"),
        True, False    # CrossFeatureVoting
    ),
    (
        "strategy5_LR_XGB_full_SVC_XGB_opt",
        joblib.load(f"{ensemble_dir}/cross_ensemble_lr_xgb_full_svc_xgb_opt.pkl"),
        True, False    # CrossFeatureVoting
    ),
]

for name, _, _, _ in benchmarks:
    print(f"   {name}")

# =========================================================
# 4. FEATURE EXTRACTION — run ONCE, reused by all 5 strategies
# =========================================================
print("\n" + "="*60)
print(f" STEP 4: FEATURE EXTRACTION [{str(device).upper()}]")
print("="*60)

# Train set — for ms/sample timing only
print(f"Extracting train features ({N_train} samples)...")
start_ext_full = time.time()
with patch('torch.load', gpu_torch_load):
    X_ext_full_train = feature_combiner_full.transform(X_train)
t_ext_full_per   = (time.time() - start_ext_full) * 1000 / N_train
X_ext_full_train = to_numpy(X_ext_full_train)

start_ext_opt = time.time()
_ = X_ext_full_train[:, keep_indices]
t_ext_opt_per = (time.time() - start_ext_opt) * 1000 / N_train
t_ext_per     = t_ext_full_per + t_ext_opt_per

print(f"Full feature extraction   : {t_ext_full_per:.6f} ms/sample")
print(f"Opt slicing (keep_indices): {t_ext_opt_per:.6f} ms/sample")
print(f"Combined extraction       : {t_ext_per:.6f} ms/sample")

# Test set — used for all inference runs
print(f"\nExtracting test features ({N_test} samples)...")
with patch('torch.load', gpu_torch_load):
    X_test_ext_full = feature_combiner_full.transform(X_test)
X_test_ext_full = to_numpy(X_test_ext_full)
X_test_ext_opt  = X_test_ext_full[:, keep_indices]

print(f"Test full shape : {X_test_ext_full.shape}")
print(f"Test opt  shape : {X_test_ext_opt.shape}")

# System stats — captured once after extraction
cpu_usage = psutil.cpu_percent(interval=1)
process   = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / (1024 * 1024)

# =========================================================
# 5. INFERENCE BENCHMARK — ALL 5 STRATEGIES
# =========================================================
print("\n" + "="*60)
print(f" STEP 5: INFERENCE BENCHMARK — 10,000 RUNS EACH")
print("="*60)

NUM_RUNS = 10000
results  = []

for model_name, model, needs_opt, use_opt_as_full in benchmarks:
    print(f"\n── {model_name} ──")

    inf_times = []
    for i in range(NUM_RUNS):
        start_inf = time.time()

        if needs_opt:
            # S3, S4, S5 — CrossFeatureVoting
            preds = model.predict(X_test_ext_full, X_test_ext_opt)
        elif use_opt_as_full:
            # S2 — PreFittedVoting trained on optimized features
            preds = model.predict(X_test_ext_opt)
        else:
            # S1 — PreFittedVoting trained on full features
            preds = model.predict(X_test_ext_full)

        inf_ms = (time.time() - start_inf) * 1000
        inf_times.append(inf_ms / N_test)

        if i % 2000 == 0 and i > 0:
            print(f"  > {i}/{NUM_RUNS} iterations done...")

    inf_per = np.mean(inf_times)
    inf_std = np.std(inf_times)
    acc     = accuracy_score(y_test, preds)

    # Feature cost per strategy:
    # S1 → full only       S2 → opt only (slicing still needed)
    # S3/S4/S5 → full + opt
    if needs_opt:
        feat_cost     = t_ext_per          # full + opt slicing
        feat_opt_col  = t_ext_opt_per
        import_opt_col = import_time_opt
    elif use_opt_as_full:
        feat_cost     = t_ext_per          # still needs full extract + slice to get opt
        feat_opt_col  = t_ext_opt_per
        import_opt_col = import_time_opt
    else:
        feat_cost     = t_ext_full_per     # full only
        feat_opt_col  = 0.0
        import_opt_col = 0.0

    print(f"  Inference : {inf_per:.6f} ms/sample  (std={inf_std:.6f})")
    print(f"  Total lat : {feat_cost + inf_per:.6f} ms/sample")
    print(f"  Accuracy  : {acc:.4f}")

    results.append({
        "Model":                               model_name,
        "Device":                              str(device),
        "Train (ms)":                          0.0,
        "Inference (ms/sample)":               round(inf_per, 6),
        "Inference Std (ms/sample)":           round(inf_std, 6),
        "Feature Extraction Full (ms/sample)": round(t_ext_full_per, 6),
        "Feature Opt Slicing (ms/sample)":     round(feat_opt_col, 6),
        "Total Latency (ms/sample)":           round(feat_cost + inf_per, 6),
        "CPU %":                               round(cpu_usage, 1),
        "Memory (MB)":                         round(memory_mb, 2),
        "Import Full Combiner (ms)":           round(import_time_full, 4),
        "Import Opt Indices (ms)":             round(import_opt_col, 4),
        "Test Accuracy":                       round(acc, 4)
    })

# =========================================================
# 6. FINAL RESULTS TABLE + SAVE
# =========================================================
print("\n" + "="*60)
print(" FINAL RESULTS — ALL 5 STRATEGIES")
print("="*60)

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

df_results.to_csv("/kaggle/working/all_strategies_computational_cost.csv", index=False)
print("\nSaved -> all_strategies_computational_cost.csv")

X_val is already a DataFrame — no reset needed.

Running on: cuda
GPU : Tesla P100-PCIE-16GB
VRAM: 17.1 GB

 STEP 1: LOADING DATA
Train samples : 22527
Test  samples : 7510

 STEP 2: LOADING FEATURE COMBINER
BERT is on    : cuda:0
Combiner load : 10700.5 ms
Indices  load : 5.7 ms

 STEP 3: LOADING ALL 5 ENSEMBLE MODELS
   strategy1_full_voting
   strategy2_opt_voting
   strategy3_LR_XGB_cross_feature
   strategy4_LR_full_SVC_XGB_opt
   strategy5_LR_XGB_full_SVC_XGB_opt

 STEP 4: FEATURE EXTRACTION [CUDA]
Extracting train features (22527 samples)...
Processing 22527 samples on cuda:0 with batch size 64...
  > Processed 8000/22527...
  > Processed 16000/22527...
Full feature extraction   : 14.140859 ms/sample
Opt slicing (keep_indices): 0.031151 ms/sample
Combined extraction       : 14.172011 ms/sample

Extracting test features (7510 samples)...
Processing 7510 samples on cuda:0 with batch size 64...
Test full shape : (7510, 7782)
Test opt  shape : (7510, 2896)

 STEP 5: INFERENCE BENCHM